# ABM playground

Select the **abm** Python environment, then **Run All**. The setup below works with the kernel started in either the MantiShrimp project root or `paper/section_1`; no working-directory change or package installation is needed in the existing abm environment.

Edit the **Tunable inputs** cell and run that cell and everything below it again. For animation-only edits, rerun the inputs cell and the animation cell; the box and live-cell radii still come from the completed simulation; dead-target radius is an independent display setting. The original Hookean force/motility settings are retained; `SIM_DURATION` starts at **5** for a shorter preview (previously 25). For a very quick check, try 5 killers, 20 targets, and duration 2. More cells, longer duration, or smaller `DT0`/`DS` take longer.

This notebook uses the original `script/simulator` model. If using a fresh Python environment, install its notebook dependencies once with `%pip install numpy pandas tqdm matplotlib ipython pillow ipykernel`, then restart the kernel.

In [ ]:
from pathlib import Path
import sys
import time
import scienceplots

# Find the checkout from either the project root or a nested notebook folder.
PROJECT_ROOT = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "script/simulator/simulator.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start this notebook inside the MantiShrimp checkout.")
SIMULATOR_DIR = PROJECT_ROOT / "script" / "simulator"
if str(SIMULATOR_DIR) not in sys.path:
    sys.path.insert(0, str(SIMULATOR_DIR))

import numpy as np
import matplotlib.pyplot as plt
from simulator import KTSimulator
from virsualisation import animate_KT
plt.style.use('science')

NOTEBOOK_DIR = PROJECT_ROOT / "paper" / "section_1"
plt.rcParams.update({
    # "mathtext.fontset": "stix",
    # "font.family": "STIXGeneral",
    "legend.fontsize": 20,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "axes.titlesize": 22,
    "axes.labelsize": 22,
})
print(f"Project: {PROJECT_ROOT}\nPython: {sys.executable}")

## Tunable inputs

All **50** `KTSimulator` arguments are listed below. Lengths and times use the simulator's own units; keep rates consistent with them. `KK`, `TT`, and `KT` mean killer–killer, target–target, and killer–target.

- **Population/domain:** change cell counts and box limits to vary density. `periodic` wraps cells at the box edges; `confined` reflects them. Keep initial positions inside the box. If changing to an off-centre box, supply `init_cell_bound` explicitly; the legacy `None` default multiplies each box coordinate by 0.9.
- **Initial layout:** `region_shape` accepts `square`, `rectangle`, or `circle`; `init_mode` accepts `uniform` (Poisson disk) or `gaussian`. Circle mode uses `ini_cell_centre` and `ini_cell_region_radius`. `init_sigma=None` chooses a region-based Gaussian width. If cells cannot fit, enlarge the region, reduce counts/radii, or reduce `min_distance_factor` (keep it at least 1 for non-overlapping starts).
- **Initial probability/damage:** `KILL_PROB_INIT` and `DEATH_FAC_INIT` accept a scalar, `(mean, sd)`, or `None`. `None` samples uniformly from 0.1–0.9 and 0–0.1, respectively. Normal draws for these two inputs are **not clipped** in this legacy model; choose values accordingly. Per-cell arrays are not supported.
- **Initial states:** `KILLER_STATE_INIT` and `TARGET_STATE_INIT` additionally accept mixtures, e.g. `[(0.5, 0.2), (0.5, 1.0)]` for equal low/high-state populations. A tuple `(0.8, 0.1)` samples clipped normal states; `None` samples uniformly from 0–1. Keep scalar states in 0–1.
- **Killing:** `RECOVERY=True` accumulates damage and applies `RECOVERY_SPEED`; set that speed to 0 for no repair. `RECOVERY=False` switches to **instant killing** on a successful contact. To keep killing probability fixed at 1, use `KILL_PROBABILISTIC=False` and `kill_prob_mode="constant"`; state exhaustion is controlled separately by the two state-decay rates.
- **Seeds:** change `SEED` for a new reproducible run, or set it to `None` for a fresh random run. The two seed arguments can also be edited separately. NumPy is seeded before initialization as well as the simulator's own dynamics seeding.

`F_A_KK`, `F_A_TT`, and `DISAPPEAR_DELAY` are accepted legacy arguments with **no effect** in the current implementation. Capture distance (1.5× radii sum), contact tolerance (0.05× radii sum), and the killing-rate range (0–1) are fixed inside the model, not exposed inputs.

In [ ]:
SEED = 42  # Integer in [0, 2**32 - 1], or None for a fresh random run.

params = {
    # Population and cell size
    "N_KILLER": 10,
    "N_TARGET": 40,
    "KILLER_RADIUS": 12.5,
    "TARGET_RADIUS": 15.0,

    # Time and randomness
    "SIM_DURATION": 15.0, 
    "DT0": 1 / 25,        # Maximum integration timestep.
    "DS": 1.0,            # Adaptive timestep scale; smaller values mean more steps.
    "RNG_SEED": SEED,     # Initial cell properties, motion, binding and killing.
    "ini_seed": SEED,     # Initial spatial sampling.

    # Domain
    "BOX_X_MIN": -600.0,
    "BOX_X_MAX": 600.0,
    "BOX_Y_MIN": -600.0,
    "BOX_Y_MAX": 600.0,
    "boundary_condition": "confined",  # "periodic" or "confined"

    # Initial layout
    "region_shape": "square",  # "square", "rectangle", or "circle"
    "init_mode": "uniform",    # "uniform" or "gaussian"
    "init_sigma": None,        # Gaussian spread; None uses the region-based default.
    "init_cell_bound": None,   # (xmin, xmax, ymin, ymax); None uses 0.9 * box coordinates.
    "ini_cell_centre": (0.0, 0.0),   # Used only for circle initialization.
    "ini_cell_region_radius": 300.0, # Used only for circle initialization.
    "min_distance_factor": 4,      # Initial spacing multiplier.

    # Motion: polarity relaxation/noise, motility, and positional noise
    "KILLER_POL_DECAY": 2.5,
    "TARGET_POL_DECAY": 5,
    "KILLER_POL_NOISE": 2.0,
    "TARGET_POL_NOISE": 0.75,
    "KILLER_MOTILITY": 175.0,
    "TARGET_MOTILITY": 125.0,
    "KILLER_TRANS_NOISE": 0.75,
    "TARGET_TRANS_NOISE": 0.2,

    # Hookean overlap repulsion, adhesion and binding rates
    "KREP_KT": 250.0,
    "KREP_TT": 250.0,
    "KREP_KK": 250.0,
    "F_A_KT": 25.0,
    "F_A_TT": 1.0,  # Inactive legacy argument: TT pairs do not bind.
    "F_A_KK": 1.0,  # Inactive legacy argument: KK pairs do not bind.
    "K_BIND": 3.0,
    "K_UNBIND": 0.5,

    # Killing probability and killer exhaustion
    "KILL_PROBABILISTIC": True,
    "KILL_PROB_INIT": 1.0,    # None, a scalar probability, or (mean, sd).
    "kill_prob_mode": "constant", # "constant" or "decay"
    "DECAY_PER_CONTACT": 0.05,
    "KILLER_STATE_INIT": 1.0,  # Scalar, None, (mean, sd), or a weighted state mixture.
    "MIN_STATE_DECAY_dt": 0.001,
    "MAX_STATE_DECAY_dt": 0.1,

    # Target susceptibility, damage and repair
    "TARGET_STATE_INIT": 1.0,
    "DEATH_FAC_INIT": None,    # None, a scalar damage, or (mean, sd).
    "KILL_DEATH_THRESHOLD": 1.0,
    "RECOVERY": False,         # False switches to instant successful-contact killing.
    "RECOVERY_SPEED": 0.0,
    "DISAPPEAR_DELAY": 0.5,    # Inactive legacy argument: dead targets remain visible.
}

# Display/export controls. Cell sizes, domain and boundary follow the completed run.
PLOT_MARGIN = 0.06
ANIMATION_EMBED_LIMIT_MB = 100
animation_params = {
    "dead_target_radius": params["TARGET_RADIUS"],
    "killer_colour": "#92C4E9",
    "live_target_colour": "#96C36E",
    "dead_target_colour": "#EA9A9D",
    "traj_steps": 250,          # Number of simulated steps retained in each trail.
    "traj_linewidth": 2,
    "traj_alpha": 0.45, 
    "cir_linewidth": 1.5,
    "figsize": (6, 7),
    "fig_dpi": 100,
    "interval": 40,            # Playback milliseconds per frame, not simulation time.
    "frame_step": 10,          # Render every nth simulation step; 1 shows all steps.
    "add_swatch_legend": True,
    "legend_labels": None,     # Or three labels: [killer, live target, dead target].
    "render_html": True,
    "save_gif_path": None,     # Optional: "outputs/vis_test.gif", relative to this notebook.
    "gif_fps": 25,             # GIF playback speed; independent of simulation time.
}

## Run the simulation

Rerun this cell after changing model parameters. The checks catch common invalid settings before a long run starts. The summary and history table remain available for analysis; the simulator may finish its last recorded step just before `SIM_DURATION`.

In [ ]:
# Guard settings that can produce empty runs, invalid geometry or zero timesteps.
for name in ("N_KILLER", "N_TARGET"):
    if isinstance(params[name], bool) or not isinstance(params[name], (int, np.integer)) or params[name] <= 0:
        raise ValueError(f"{name} must be a positive integer.")
for name in ("KILLER_RADIUS", "TARGET_RADIUS", "SIM_DURATION", "DT0", "DS",
             "KILL_DEATH_THRESHOLD", "min_distance_factor"):
    if not np.isfinite(params[name]) or params[name] <= 0:
        raise ValueError(f"{name} must be finite and positive.")
for axis in ("X", "Y"):
    low, high = params[f"BOX_{axis}_MIN"], params[f"BOX_{axis}_MAX"]
    if not np.isfinite([low, high]).all() or high - low <= 2 * max(params["KILLER_RADIUS"], params["TARGET_RADIUS"]):
        raise ValueError(f"The {axis} box limits must leave room for the largest cell.")
for name, choices in {
    "boundary_condition": ("periodic", "confined"),
    "region_shape": ("square", "rectangle", "circle"),
    "init_mode": ("uniform", "gaussian"),
    "kill_prob_mode": ("constant", "decay"),
}.items():
    if params[name] not in choices:
        raise ValueError(f"{name} must be one of {choices}.")
for name in ("KILLER_POL_DECAY", "TARGET_POL_DECAY", "KILLER_POL_NOISE", "TARGET_POL_NOISE",
             "KILLER_MOTILITY", "TARGET_MOTILITY", "KILLER_TRANS_NOISE", "TARGET_TRANS_NOISE",
             "KREP_KK", "KREP_TT", "KREP_KT", "F_A_KT", "K_BIND", "K_UNBIND",
             "DECAY_PER_CONTACT", "MIN_STATE_DECAY_dt", "MAX_STATE_DECAY_dt", "RECOVERY_SPEED"):
    if not np.isfinite(params[name]) or params[name] < 0:
        raise ValueError(f"{name} must be finite and non-negative.")
if params["MIN_STATE_DECAY_dt"] > params["MAX_STATE_DECAY_dt"]:
    raise ValueError("MIN_STATE_DECAY_dt must not exceed MAX_STATE_DECAY_dt.")
if params["init_sigma"] is not None and (not np.isfinite(params["init_sigma"]) or params["init_sigma"] <= 0):
    raise ValueError("init_sigma must be None or finite and positive.")
if params["region_shape"] == "circle":
    centre, radius = params["ini_cell_centre"], params["ini_cell_region_radius"]
    if centre is None or np.shape(centre) != (2,) or not np.isfinite(centre).all():
        raise ValueError("Circle initialization needs a finite (x, y) ini_cell_centre.")
    if radius is None or not np.isfinite(radius) or radius <= 0:
        raise ValueError("Circle initialization needs a positive ini_cell_region_radius.")
if params["init_cell_bound"] is not None:
    bounds = np.asarray(params["init_cell_bound"], dtype=float)
    if bounds.shape != (4,) or not np.isfinite(bounds).all() or bounds[0] >= bounds[1] or bounds[2] >= bounds[3]:
        raise ValueError("init_cell_bound must be finite (xmin, xmax, ymin, ymax) with increasing limits.")

# Legacy scalar initializers do not enforce probability/state ranges.
state_inputs = ("KILLER_STATE_INIT", "TARGET_STATE_INIT")
if params["KILL_PROBABILISTIC"]:
    state_inputs += ("KILL_PROB_INIT",)
for name in state_inputs:
    value = params[name]
    if np.isscalar(value) and (not np.isfinite(value) or not 0 <= value <= 1):
        raise ValueError(f"Scalar {name} must be finite and between 0 and 1.")
value = params["DEATH_FAC_INIT"]
if np.isscalar(value) and (not np.isfinite(value) or value < 0):
    raise ValueError("Scalar DEATH_FAC_INIT must be finite and non-negative.")

# The legacy simulator seeds NumPy after initialization, so seed it here too.
np.random.seed(params["RNG_SEED"])
started = time.perf_counter()
(cell_history_df, killer_positions, target_positions,
 killingProbability_history, DeathFactor_history, sim_settings) = KTSimulator(**params)
last_step = cell_history_df["step"].max()
final_targets = cell_history_df[
    (cell_history_df["step"] == last_step) & (cell_history_df["cell_type"] == "target")
]
print(f"Finished in {time.perf_counter() - started:.1f}s; {last_step + 1} recorded steps.")
print(f"Targets alive: {int(final_targets['alive_status'].sum())}/{params['N_TARGET']}")
cell_history_df.head()

## Play / export

Use the playback controls below. Increase `frame_step` or reduce `fig_dpi` for a lighter animation. To save a GIF, set `save_gif_path` in the input cell to e.g. `"outputs/vis_test.gif"`; relative paths always resolve beside this notebook, regardless of where the kernel started. Leave it as `None` to preview without writing a file. For export only, set `render_html=False`.

The animation shows recorded steps at a fixed playback interval, so adaptive simulation timesteps do not imply real-time playback.

In [ ]:
sim_box = tuple(sim_settings[key] for key in ("BOX_X_MIN", "BOX_X_MAX", "BOX_Y_MIN", "BOX_Y_MAX"))
xwidth, ywidth = sim_box[1] - sim_box[0], sim_box[3] - sim_box[2]
ax_box = (
    sim_box[0] - PLOT_MARGIN * xwidth, sim_box[1] + PLOT_MARGIN * xwidth,
    sim_box[2] - PLOT_MARGIN * ywidth, sim_box[3] + PLOT_MARGIN * ywidth,
)
plot_options = animation_params.copy()
if plot_options["save_gif_path"] is not None:
    gif_path = Path(plot_options["save_gif_path"]).expanduser()
    if not gif_path.is_absolute():
        gif_path = NOTEBOOK_DIR / gif_path
    gif_path.parent.mkdir(parents=True, exist_ok=True)
    plot_options["save_gif_path"] = str(gif_path)
plt.rcParams["animation.embed_limit"] = ANIMATION_EMBED_LIMIT_MB
animation = animate_KT(
    cell_history_df, killer_positions, target_positions,
    killer_radius=sim_settings["KILLER_RADIUS"],
    target_radius=sim_settings["TARGET_RADIUS"],
    box=sim_box, ax_box=ax_box,
    boundary_condition=sim_settings["boundary_condition"],
    **plot_options,
)

In [ ]:
# Run after the animation cell.
from IPython.display import SVG, display

FRAME_STEP = 350  # Simulation step, not the animation's displayed-frame index.
TRANSPARENT = True
SVG_PATH = NOTEBOOK_DIR / "outputs" / f"frame_{FRAME_STEP:04d}.svg"

if FRAME_STEP not in cell_history_df["step"].values:
    raise ValueError(
        f"Choose a step between 0 and {cell_history_df['step'].max()}."
    )

# Reuse the animation's drawing function, including colours and trails.
animation._func(FRAME_STEP)
SVG_PATH.parent.mkdir(parents=True, exist_ok=True)

with plt.rc_context({"svg.fonttype": "none"}):  # Keep text editable.
    animation._fig.savefig(
        SVG_PATH,
        format="svg",
        bbox_inches="tight",
        transparent=TRANSPARENT,
    )

display(SVG(filename=str(SVG_PATH)))
print(f"Saved: {SVG_PATH}")